# Predicting LLM bias

In *Languages as distributions* and *From characters to tokens*, we built two sets of pairwise distance matrices between 8 languages — one from character frequency distributions, one from token frequency distributions. Both recovered the structure of language families from raw statistical properties of text, with no linguistic knowledge injected.

We now ask the central question of this project:

> **Do these statistical distances predict the distances between languages in the embedding space of a multilingual language model?**

If they do, it would mean that the geometry a model learns during training is at least partially determined by the raw statistical structure of the languages it sees — and that this structure is measurable from text alone, before any model is involved.

This has a direct implication for LLM bias. A multilingual model like XLM-R is known to represent some languages better than others — to be closer to some languages in its internal geometry, and farther from others. If that geometry correlates with statistical distances, then the bias is not arbitrary: it is structurally driven by the distributional properties of the languages themselves.

## 1. Setup & loading

In [1]:
import sys
sys.path.append('../languages-distributions')
sys.path.append('../token-geometry')
sys.path.append('../llm-bias')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import MDS
from matplotlib.patches import Patch

from flores_loader import load_flores
from embeddings import load_model, get_all_language_embeddings, cosine_distance_matrix
from mantel import mantel_test, mantel_summary
from token_frequencies import load_tokenizer, token_frequencies
from frequencies import unigram_frequencies, ngram_frequencies
from distance_matrix import compute_distance_matrix
import viz_bias as viz

print("Loading Flores-200...")
flores_texts = load_flores(split='devtest')
print(f"\nLanguages: {list(flores_texts.keys())}")
print(f"Sentences per language: {len(list(flores_texts.values())[0])}")

c:\Users\comer\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Flores-200...
Loading opus-100...


c:\Users\comer\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\comer\.cache\huggingface\hub\datasets--Helsinki-NLP--opus-100. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating validation split: 100%|██████████| 2000/2000 [00:00<00:00, 119103.92 examples/s

  French: 2000 sentences loaded


ValueError: BuilderConfig 'en-de' not found. Available: ['af-en', 'am-en', 'an-en', 'ar-de', 'ar-en', 'ar-fr', 'ar-nl', 'ar-ru', 'ar-zh', 'as-en', 'az-en', 'be-en', 'bg-en', 'bn-en', 'br-en', 'bs-en', 'ca-en', 'cs-en', 'cy-en', 'da-en', 'de-en', 'de-fr', 'de-nl', 'de-ru', 'de-zh', 'dz-en', 'el-en', 'en-eo', 'en-es', 'en-et', 'en-eu', 'en-fa', 'en-fi', 'en-fr', 'en-fy', 'en-ga', 'en-gd', 'en-gl', 'en-gu', 'en-ha', 'en-he', 'en-hi', 'en-hr', 'en-hu', 'en-hy', 'en-id', 'en-ig', 'en-is', 'en-it', 'en-ja', 'en-ka', 'en-kk', 'en-km', 'en-kn', 'en-ko', 'en-ku', 'en-ky', 'en-li', 'en-lt', 'en-lv', 'en-mg', 'en-mk', 'en-ml', 'en-mn', 'en-mr', 'en-ms', 'en-mt', 'en-my', 'en-nb', 'en-ne', 'en-nl', 'en-nn', 'en-no', 'en-oc', 'en-or', 'en-pa', 'en-pl', 'en-ps', 'en-pt', 'en-ro', 'en-ru', 'en-rw', 'en-se', 'en-sh', 'en-si', 'en-sk', 'en-sl', 'en-sq', 'en-sr', 'en-sv', 'en-ta', 'en-te', 'en-tg', 'en-th', 'en-tk', 'en-tr', 'en-tt', 'en-ug', 'en-uk', 'en-ur', 'en-uz', 'en-vi', 'en-wa', 'en-xh', 'en-yi'

In [ ]:
print("Loading XLM-R...")
tokenizer, model = load_model()
print("Model loaded.")

## 2. Why Flores-200?

To compare how XLM-R represents different languages, we need to extract embeddings and compute distances between them. But there is a fundamental problem: if we pass different texts in different languages, we cannot know whether the differences in embedding space come from the **language** or from the **content**.

Imagine comparing an embedding of a French cooking recipe with an embedding of a Polish legal document. Any distance we measure would reflect both the linguistic difference and the content difference — we cannot disentangle the two.

The solution is **parallel corpora**: the exact same sentences, translated into all languages. If the content is held constant, any remaining variation in embedding space must come from the language itself.

**Flores-200** (Meta AI, 2022) is the standard benchmark for this. It contains 1012 sentences drawn from English Wikipedia, professionally translated into 200 languages. Every sentence has an exact counterpart in every language — making it the ideal tool for measuring how a multilingual model represents different languages under controlled conditions.

Our procedure:
1. Pass each of the 1012 sentences through XLM-R for each language
2. Extract the sentence embedding (mean pooling over token representations)
3. Average all 1012 embeddings to get a single vector representing each language
4. Compute pairwise cosine distances between these 8 vectors

## 3. Extracting embeddings

XLM-R produces a 768-dimensional vector for each token in the input. To get a single vector for a sentence, we use **mean pooling** — we average the vectors of all tokens in the sentence. This is a standard and robust approach.

We then average over all 1012 Flores sentences to get a single vector per language. This mean vector represents the "center of mass" of the language in XLM-R's embedding space.

In [ ]:
print("Extracting language embeddings from XLM-R (last layer)...")
print("This will take a few minutes.\n")

embeddings = get_all_language_embeddings(flores_texts, tokenizer, model, layer=-1)

print("\nDone.")
print(f"Embedding dimension: {list(embeddings.values())[0].shape[0]}")

In [ ]:
# Quick sanity check — cosine similarity between a few pairs
from numpy.linalg import norm

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

pairs = [
    ('French', 'Spanish'),
    ('French', 'Polish'),
    ('English', 'German'),
    ('English', 'Polish'),
    ('Spanish', 'Portuguese')
]

print("Cosine similarity between language embeddings:")
for l1, l2 in pairs:
    sim = cosine_similarity(embeddings[l1], embeddings[l2])
    print(f"  {l1} — {l2}: {sim:.4f}")

## 4. Distance matrix in embedding space

We now compute the full pairwise distance matrix between all 8 languages in XLM-R's embedding space. We use cosine distance (1 - cosine similarity) — the standard metric for comparing high-dimensional vectors.

This matrix is the key object of Act 3. It tells us how XLM-R internally organizes languages — which languages it considers similar, and which it considers distant. The question is whether this organization matches what we measured in Acts 1 and 2.

In [ ]:
languages, embedding_matrix = cosine_distance_matrix(embeddings)
viz.plot_embedding_matrix(languages, embedding_matrix)

## 5. MDS projection

We project the embedding distance matrix into 2D using MDS, as we did in Acts 1 and 2. This lets us visually compare the geometry of languages in XLM-R's embedding space with the geometry we measured from raw text statistics.

The key question: do the same clusters appear? Are the same language pairs close or distant?

In [ ]:
viz.plot_embedding_mds(languages, embedding_matrix)

## 6. The Mantel test

Visual comparison is informative but not rigorous. To formally test whether our statistical distance matrices predict the embedding distance matrix, we use the **Mantel test**.

### What is the Mantel test?

The Mantel test measures the correlation between two distance matrices. Given two n×n matrices, it extracts the upper triangles as flat vectors and computes the correlation between them — a high correlation means the two matrices agree on which pairs of objects are close and which are far.

But correlation alone is not enough — we need to know if it is statistically significant. The Mantel test does this through **permutation**: it randomly shuffles the rows and columns of one matrix thousands of times, recomputes the correlation each time, and builds a null distribution. The p-value is the proportion of permutations that produce a correlation as high as the observed one.

This is the standard method for comparing distance matrices in ecology, genetics, and linguistics — and exactly the right tool for our question.

### What we test

We run the Mantel test between the embedding distance matrix and:
- The character-level distance matrix from Act 1 (Hellinger, bigrams — our best performer)
- The token-level distance matrix from Act 2 (Hellinger)

If the correlations are significant, it means the statistical structure of languages — measurable from raw text — predicts how XLM-R organizes languages in its internal geometry.

In [ ]:
from text_loader import load_texts as load_texts_token
import os

# Rebuild character-level distance matrix (bigrams, Hellinger) from Act 1
# We need the original book texts for this
books_path = '../data/Books'
texts_upper = {}
for filename in os.listdir(books_path):
    if filename.endswith('.txt'):
        lang = filename.replace('.txt', '')
        with open(os.path.join(books_path, filename), 'r', encoding='utf-8') as f:
            text = f.read().replace('\n', ' ').upper()
        texts_upper[lang] = text

char_dists = {lang: ngram_frequencies(text, 2) for lang, text in texts_upper.items()}
char_langs, char_matrix = compute_distance_matrix(char_dists, metric='hellinger')
print("Character bigram matrix built.")

# Rebuild token-level distance matrix (Hellinger) from Act 2
texts_lower = load_texts_token(books_path)
token_dists = {
    lang: token_frequencies(text[:500_000], tokenizer)
    for lang, text in texts_lower.items()
}
token_langs, token_matrix = compute_distance_matrix(token_dists, metric='hellinger')
print("Token matrix built.")

In [ ]:
# Align all matrices to the same language order
def reorder_matrix(matrix, source_langs, target_langs):
    idx = [source_langs.index(l) for l in target_langs]
    return matrix[np.ix_(idx, idx)]

ref_langs = languages  # embedding matrix language order
char_matrix_aligned = reorder_matrix(char_matrix, char_langs, ref_langs)
token_matrix_aligned = reorder_matrix(token_matrix, token_langs, ref_langs)

print("Matrices aligned to common language order:")
print(ref_langs)

## 7. Results

We now run the Mantel test for each comparison and visualize the results.

In [ ]:
print("Running Mantel tests (999 permutations each)...\n")

r_char, p_char, perm_char = mantel_test(char_matrix_aligned, embedding_matrix, n_permutations=999)
mantel_summary('Character bigrams (Hellinger)', 'XLM-R embeddings', r_char, p_char)

print()

r_token, p_token, perm_token = mantel_test(token_matrix_aligned, embedding_matrix, n_permutations=999)
mantel_summary('Token level (Hellinger)', 'XLM-R embeddings', r_token, p_token)

In [ ]:
# Null distribution plots
viz.plot_mantel_distribution(r_char, perm_char, 'Character bigrams', 'XLM-R embeddings')
viz.plot_mantel_distribution(r_token, perm_token, 'Token level', 'XLM-R embeddings')

In [ ]:
# Summary bar chart
mantel_results = {
    'Character bigrams\nvs embeddings': (r_char, p_char),
    'Token level\nvs embeddings': (r_token, p_token)
}
viz.plot_mantel_summary(mantel_results)

## 8. Conclusions

### What we found

We measured the correlation between two types of distance matrices — built from raw text statistics — and the distance matrix extracted from XLM-R's embedding space on parallel data.

The Mantel test tells us whether this correlation is statistically significant, i.e., whether it is unlikely to have occurred by chance. A significant positive correlation would mean that the statistical structure of languages, as measured from character or token frequencies, predicts how XLM-R internally organizes those languages.

### What this implies

If the correlations are significant, the implication is important: **LLM bias is not arbitrary**. The distances between languages in a multilingual model's embedding space are at least partially driven by the raw distributional properties of the languages themselves — properties that are measurable without any model, from text alone.

This does not mean that statistical distance is the only driver of bias. Training data volume, tokenizer design, and fine-tuning choices all play a role. But it suggests that a part of the bias is structural — baked into the statistical nature of the languages before any model sees them.

### Limitations and next steps

- We worked with 8 languages — a small sample. Extending to more typologically diverse languages (agglutinative, tonal, low-resource) would strengthen or challenge these findings.
- We used a single model (XLM-R base) and a single layer. Different models and different layers may tell a different story.
- Correlation is not causation. A significant Mantel test shows that the two matrices agree — it does not prove that statistical distance *causes* the embedding geometry.

These are the natural next steps for this line of research.